# Note: exploratory notebook, not used 

# Dataset State Matrix

Two metrics are computed per cell type per dataset, measuring different failure modes.

---

## Normalized Shannon Entropy (fragmentation)

Asks: **how spread is this cell type across Leiden clusters?**

```
Cell type: B cell

Low entropy (good)          High entropy (fragmented)
───────────────────         ───────────────────────────
cluster 0 ██████████ 90%    cluster 0 ███ 20%
cluster 1 █ 10%             cluster 1 ███ 18%
                            cluster 2 ███ 17%
                            cluster 3 ███ 17%
                            cluster 4 ███ 15%
                            cluster 5 ██  13%

entropy ≈ 0.14              entropy ≈ 0.99
```

Range: `[0, 1]`. High value means the cell type bleeds across many clusters -- possibly a sign of poor resolution, annotation noise, or a heterogeneous population.

---

## KL Divergence (coherence)

Asks: **does this cell type concentrate in clusters more than chance would predict?**

```
Cell type: B cell   Global cluster distribution: all clusters roughly equal (17%)

Low KL divergence (bad)             High KL divergence (good)
─────────────────────────────────   ────────────────────────────────
cluster 0: p(c|B) 20% vs p(c) 17%  cluster 3: p(c|B) 85% vs p(c) 17%
cluster 1: p(c|B) 18% vs p(c) 17%  cluster 0: p(c|B) 15% vs p(c) 17%
...B cells match global dist...     ...B cells heavily prefer cluster 3...

KL ≈ 0.06 bits                      KL ≈ 1.2 bits
```

Range: `[0, ∞)` in bits. Computed as D_KL(p(cluster|cell_type) || p(cluster)). Near zero means the cell type mirrors the global distribution; higher means it concentrates distinctively.

---

## Reading them together

| Normalized Shannon Entropy | KL Divergence | Interpretation |
|---------|-------------|----------------|
| Low     | High        | Tight, coherent mapping -- ideal |
| Low     | Low         | Concentrated in a cluster, but that cluster is dominant globally -- not distinctive |
| High    | High        | Spread across clusters that are all globally rare -- unusual |
| High    | Low         | Fragmented and indistinct -- worst case |

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from IPython.display import display
from scipy.stats import entropy as scipy_entropy

In [ ]:
a1 = sc.read_h5ad(Path().resolve().parent / "output/clustering/data/SRX12366723_clustered.h5ad")
a2 = sc.read_h5ad(Path().resolve().parent / "output/clustering/data/SRX13198730_clustered.h5ad")
a3 = sc.read_h5ad(Path().resolve().parent / "output/clustering/data/SRX17412841_clustered.h5ad")

datasets = {"SRX12366723": a1, "SRX13198730": a2, "SRX17412841": a3}

In [ ]:
fig, axs = plt.subplots(len(datasets), 2, figsize=(14, 6 * len(datasets)))

for row, (srx, a) in enumerate(datasets.items()):
    sc.pl.umap(a, color="cell_type", ax=axs[row, 0], show=False, title=f"{srx}: cell_type")
    sc.pl.umap(a, color="leiden_merged", ax=axs[row, 1], show=False, title=f"{srx}: leiden_merged")

plt.tight_layout()
plt.show()

In [ ]:
nse_rows = {}
kld_rows = {}

for srx, a in datasets.items():
    global_labels = a.obs["leiden_merged"]
    all_clusters = global_labels.cat.categories
    q = global_labels.value_counts(normalize=True).reindex(all_clusters, fill_value=0)
    nse_row = {}  # normalized shannon entropy
    kld_row = {}  # kl divergence
    for cell_type in a.obs["cell_type"].unique():
        labels = a.obs[a.obs["cell_type"] == cell_type]["leiden_merged"]
        counts = labels.value_counts(normalize=True)
        counts = counts[counts > 0]
        nse_row[cell_type] = scipy_entropy(counts, base=2) / np.log2(len(counts)) if len(counts) > 1 else 0.0
        p = labels.value_counts(normalize=True).reindex(all_clusters, fill_value=0)
        kld_row[cell_type] = scipy_entropy(p, q, base=2)
    nse_rows[srx] = nse_row
    kld_rows[srx] = kld_row

nse_df = pd.DataFrame(nse_rows).T
nse_df.index.name = "srx"
kld_df = pd.DataFrame(kld_rows).T
kld_df.index.name = "srx"

In [ ]:
def plot_cell_type_metrics(
    summary_df: pd.DataFrame,
    sort_by: str = "normalized_shannon_entropy_mean",
    figsize: tuple[int, int] = (14, 10),
) -> None:
    """Plot entropy and KL divergence per cell type as two separate horizontal bar charts.

    Entropy is shown on its natural [0, 1] scale. KL divergence is shown on its raw scale in bits.
    Each chart is sorted by sort_by independently.

    Args:
        summary_df: DataFrame with columns normalized_shannon_entropy_mean and kl_divergence_mean, indexed by cell type.
        sort_by: column to sort rows by before plotting.
        figsize: matplotlib figure size.
    Returns:
        None
    """
    sorted_df = summary_df.sort_values(sort_by)
    labels = sorted_df.index.tolist()
    y = np.arange(len(labels))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)

    ax1.barh(y, sorted_df["normalized_shannon_entropy_mean"], color="steelblue")
    ax1.set_yticks(y)
    ax1.set_yticklabels(labels)
    ax1.set_xlabel("normalized Shannon entropy [0, 1]")
    ax1.set_title("Normalized Shannon Entropy (↑ = more fragmented)")

    ax2.barh(y, sorted_df["kl_divergence_mean"], color="darkorange")
    ax2.set_yticks(y)
    ax2.set_yticklabels([])
    ax2.set_xlabel("KL divergence (bits)")
    ax2.set_title("KL Divergence (↑ = more coherent)")

    plt.tight_layout()
    plt.show()

In [ ]:
print("Normalized Shannon Entropy (↑ = more fragmented)")
display(nse_df.round(3))

print("KL Divergence (↑ = more coherent)")
display(kld_df.round(3))

summary_df = pd.DataFrame(
    {
        "n_datasets": nse_df.notna().sum(),
        "normalized_shannon_entropy_mean": nse_df.mean(),
        "kl_divergence_mean": kld_df.mean(),
    }
)
summary_df.index.name = "cell_type"

print("Cross-dataset summary (mean per cell type)")
display(summary_df.sort_values("normalized_shannon_entropy_mean", ascending=False).round(3))

plot_cell_type_metrics(summary_df)

### Inspect cell types on full dataset run

In [ ]:
summary_csv = Path("../output/annotation_pipeline/20260509_214237/cell_type_summary.csv")
df = pd.read_csv(summary_csv).dropna(subset=["cell_type"]).set_index("cell_type")
cutoff = 10  # df["n_datasets"].describe()["25%"]
print(f"Filtered cell types appearing in fewer than {cutoff} datasets: {df[df['n_datasets'] < cutoff].index.tolist()}")
df = df[df["n_datasets"] >= cutoff]
df.sort_values("kl_divergence_mean", ascending=False)

In [ ]:
def save_metric_plot(
    summary_df: pd.DataFrame,
    output_path: Path | None = None,
    sort_by: str = "normalized_shannon_entropy_mean",
    dpi: int = 150,
) -> plt.Figure:
    """Save a two-panel horizontal bar chart of NSE and KLD means per cell type.

    The left panel shows normalized Shannon entropy (higher = more fragmented).
    The right panel shows mean KL divergence (higher = more coherent / distinct
    from the global cluster background). Cell types are sorted by sort_by ascending
    so the most fragmented cell type appears at the top.

    Args:
        summary_df: DataFrame produced by build_metric_dataframes, indexed by cell
            type, with columns normalized_shannon_entropy_mean and kl_divergence_mean.
        output_path: Destination path for the PNG file.
        sort_by: Column in summary_df used to sort cell types (default:
            normalized_shannon_entropy_mean).
        dpi: Resolution of the saved image (default: 150).
    """
    sorted_df = summary_df.sort_values(sort_by)
    labels = sorted_df.index.tolist()
    y = np.arange(len(labels))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, max(6, len(labels) * 0.5)))

    ax1.barh(y, sorted_df["normalized_shannon_entropy_mean"], color="steelblue")
    ax1.set_yticks(y)
    ax1.set_yticklabels(labels)
    ax1.set_xlabel("normalized Shannon entropy [0, 1]")
    ax1.set_title("Normalized Shannon Entropy (↑ = more fragmented)")

    ax2.barh(y, sorted_df["kl_divergence_mean"], color="darkorange")
    ax2.set_yticks(y)
    ax2.set_yticklabels([])
    ax2.set_xlabel("KL divergence (bits)")
    ax2.set_title("KL Divergence (↑ = more coherent)")

    plt.tight_layout()
    if output_path:
        output_path.mkdir(parents=True, exist_ok=True)
        fig.savefig(output_path, dpi=dpi)
    plt.show()


save_metric_plot(df)